In [0]:
# ==========================================================
# Smart Patient Readmission Risk Pipeline
# Helper Functions
# ==========================================================

import random
from datetime import datetime, timedelta

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
%run ../config/config.py

In [0]:
def apply_spark_optimizations(spark):
    """
    Apply Spark configurations.
    Skip unsupported configurations in Databricks Free Edition.
    """

    print("Applying Spark optimizations...")

    for key, value in SPARK_OPTIMIZATIONS.items():

        try:
            spark.conf.set(key, value)
            print(f"✓ Applied: {key}")

        except Exception:
            print(f"⚠ Skipped: {key}")

    print("✅ Spark optimization step completed.")

In [0]:
def write_delta_overwrite(df: DataFrame, table_name: str):
    """
    Overwrite a Delta table.
    """

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"✅ {table_name} written successfully.")

In [0]:
def read_table(table_name: str):
    """
    Read a Delta table.
    """

    return spark.table(table_name)

In [0]:
def generate_phone():

    return "9" + "".join(
        random.choice("0123456789")
        for _ in range(9)
    )

In [0]:
def weighted_choice(weight_dict):

    choices = list(weight_dict.keys())

    weights = list(weight_dict.values())

    return random.choices(
        choices,
        weights=weights,
        k=1
    )[0]

In [0]:
def log_table_stats(df: DataFrame, table_name: str):

    print("=" * 60)
    print(f"Table : {table_name}")
    print(f"Rows  : {df.count()}")
    print(f"Cols  : {len(df.columns)}")
    print("=" * 60)

In [0]:
def validate_no_duplicates(df: DataFrame, column_name: str):

    duplicate_count = (
        df.groupBy(column_name)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    if duplicate_count == 0:
        print(f"✅ No duplicates found in {column_name}")

    else:
        print(f"⚠️ {duplicate_count} duplicate values found in {column_name}")

In [0]:
def null_summary(df: DataFrame):

    expr = [

        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)

        for c in df.columns

    ]

    df.select(expr).show()